# Lesson 2.3 — Distributing Tools Across Agents & Configuring tool_choice

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Explain why too many tools reduce selection reliability
- Scope each subagent's tools to its role and add limited cross-role tools
- Replace generic tools with constrained alternatives
- Configure tool_choice: auto, any, and forced selection

**You will produce:** A per-role tool map plus tool_choice configs (any, and forced extract_metadata first).

## Concepts

**More tools = harder, less reliable selection.** Agents given tools outside their specialization tend to misuse them (a synthesis agent attempting web searches). Apply **scoped tool access**: give each agent only what its role needs, plus a few limited **cross-role tools** for high-frequency needs.

Sample Question 9: the synthesis agent needs frequent simple fact-checks. The best fix is a **scoped `verify_fact` tool** for the 85% simple cases while routing the 15% complex verifications back through the coordinator - least privilege, not full web-search access.

### tool_choice
- **`auto`** - model may return text instead of calling a tool.
- **`any`** - model MUST call a tool but can choose which (guarantees structured output).
- **forced** - `{"type": "tool", "name": "extract_metadata"}` forces a specific tool (e.g. run extraction before enrichment), then continue in follow-up turns.

Also prefer **constrained tools** over generic ones (replace `fetch_url` with `load_document` that validates document URLs).

> **Reference:** [Anthropic - tool_choice](https://docs.anthropic.com/en/docs/build-with-claude/tool-use)

In [ ]:
# !pip install anthropic
import os, json

try:
    import anthropic
except ImportError:
    anthropic = None
    print("anthropic SDK not installed. Run: pip install anthropic")

API_KEY = os.environ.get("ANTHROPIC_API_KEY")
HAVE_API = bool(API_KEY) and anthropic is not None

client = anthropic.Anthropic() if HAVE_API else None  # reads ANTHROPIC_API_KEY

# Current model aliases (confirm latest at https://docs.anthropic.com/en/docs/about-claude/models)
MODEL_HAIKU = "claude-haiku-4-5"
MODEL_SONNET = "claude-sonnet-4-5"
MODEL_OPUS = "claude-opus-4-1"

def mock_text(prompt):
    """Offline fallback so the notebook runs without an API key."""
    return f"[MOCK RESPONSE] Set ANTHROPIC_API_KEY for a live call. Prompt was: {str(prompt)[:80]}..."

print("API ready" if HAVE_API else "No API key - cells will print mock responses")

## Code-Along: Step 1 — Scope tools per role

Each agent gets role tools; the synthesis agent gets a single scoped cross-role tool.

In [ ]:
ROLE_TOOLS = {
    "web_search": ["web_search", "load_document"],
    "doc_analysis": ["load_document", "extract_data_points"],
    "synthesis": ["verify_fact"],  # scoped cross-role tool for simple fact-checks
}
def misuse_risk(role):
    # Fewer, role-appropriate tools -> lower misuse risk
    return "low" if len(ROLE_TOOLS[role]) <= 2 else "elevated"
for r in ROLE_TOOLS:
    print(r, ROLE_TOOLS[r], '->', misuse_risk(r))

## Code-Along: Step 2 — Configure tool_choice

Pick auto, any, or forced depending on whether/which tool must run.

In [ ]:
def tool_choice(mode, name=None):
    if mode == "auto":
        return {"type": "auto"}
    if mode == "any":
        return {"type": "any"}
    if mode == "forced":
        return {"type": "tool", "name": name}
    raise ValueError(mode)

print("guarantee a tool call:", tool_choice("any"))
print("force metadata first:", tool_choice("forced", "extract_metadata"))

## Code-Along: Step 3 — Replace a generic tool with a constrained one

A validating `load_document` is safer than an unrestricted `fetch_url`.

In [ ]:
def load_document(url):
    allowed = (".pdf", ".md", ".txt", ".html")
    if not url.startswith("https://") or not url.endswith(allowed):
        return {"isError": True, "errorCategory": "validation",
                "isRetryable": False, "description": "Not a valid document URL"}
    return {"isError": False, "result": f"loaded {url}"}

print(load_document("https://example.com/report.pdf"))
print(load_document("https://example.com/arbitrary-endpoint"))

## Exercise

Implement the Sample-Question-9 routing: a `verify(claim, complex)` function where simple fact-checks use the synthesis agent's scoped `verify_fact` tool, but complex ones return a `route_to_coordinator` signal. Then build a forced-then-any sequence: force `extract_metadata`, then use `tool_choice: any` for enrichment.

In [ ]:
# TODO: implement verify(claim, complex) using ROLE_TOOLS['synthesis']
# TODO: assert complex verifications route through the coordinator
# TODO: build [tool_choice('forced','extract_metadata'), tool_choice('any')]

## Key Takeaways

- Too many tools degrade selection reliability - scope tools to each agent's role.
- Add only limited cross-role tools for high-frequency needs (e.g. verify_fact).
- tool_choice: auto (may text), any (must call some tool), forced (must call a named tool).
- Prefer constrained tools (load_document) over generic ones (fetch_url).

---

**Next:** [Lesson 2.4 — Integrating MCP Servers](./lesson-2.4-mcp-server-integration.ipynb)